# RSNA Knee Abnormality Detection — Phase 4 report labeling

Internet-off Kaggle notebook. **Do not enable internet. Do not attach a GPU.**
This notebook does **not** train an image model, does **not** create folds, and
does **not** submit to the leaderboard.

Report text stays inside this Kaggle session. It is never printed, never sent
to an external API, and never written into sanitized aggregate files.

| Path | Role |
| --- | --- |
| `/kaggle/input/rsna-knee-abnormality-detection` or `/kaggle/input/competitions/rsna-knee-abnormality-detection` | Competition data |
| Private source dataset | Allowlisted repository ZIP or unpacked `knee/` tree |
| `/kaggle/working/rsna_knee_src` | Extracted working source |
| `/kaggle/working/rsna_report_label_outputs/private` | Row-level weak labels (not for git) |
| `/kaggle/working/rsna_report_label_outputs/sanitized` | Aggregate reports only |

Software-fixture executions must keep `scientific_evaluation=false` and must
not claim scientific gold-label performance.



In [ ]:
# Bootstrap: stdlib only until the package is importable.
# Copies/extracts source under /kaggle/working. Never writes under /kaggle/input.
from __future__ import annotations

import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_report_label_outputs"
MARKERS = ("knee/config.py", "knee/report_labels.py")


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md"):
        src = src_root / filename
        if src.is_file():
            shutil.copy2(src, dest / filename)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


def locate_and_extract_source() -> Path | None:
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips = []
    dirs = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and child.suffix.lower() == ".zip" and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            if _looks_like_source(child):
                dirs.append(child)
            for nested in sorted(p for p in child.iterdir() if p.is_dir()):
                if _looks_like_source(nested):
                    dirs.append(nested)
            zips.extend(
                zip_path for zip_path in sorted(child.glob("*.zip")) if _zip_has_source(zip_path)
            )
    if zips:
        _safe_extract(zips[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        for nested in sorted(p for p in SRC_DST.iterdir() if p.is_dir()):
            if _looks_like_source(nested):
                return nested
    if dirs:
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
if SOURCE_ROOT is not None:
    resolved = str(SOURCE_ROOT.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"source_root={SOURCE_ROOT}")
else:
    print("source_root=not_found (will rely on an already importable knee package)")

try:
    from knee.kaggle_runtime import detect_kaggle_paths, KaggleLayout
    from knee.report_labels import KAGGLE_REPORT_OUTPUT_DIRNAME, run_notebook_report_labeling
except ImportError as exc:
    raise SystemExit(
        "LABELING STOPPED: could not import knee. Attach the private source dataset "
        f"(zip or unpacked tree with knee/report_labels.py). Original error: {exc}"
    ) from exc

print("internet_used=False gpu_required=False training_run=False")
print(detect_kaggle_paths(KaggleLayout(output_dirname=KAGGLE_REPORT_OUTPUT_DIRNAME)))


In [ ]:
# Preflight + report labeling. Do not print Report text or identifiers.
from pathlib import Path

from knee.kaggle_runtime import KaggleLayout, preflight
from knee.report_labels import KAGGLE_REPORT_OUTPUT_DIRNAME, run_notebook_report_labeling

layout = KaggleLayout(output_dirname=KAGGLE_REPORT_OUTPUT_DIRNAME)
DATA_ROOT = Path(os.environ["RSNA_DATA_ROOT"]) if os.environ.get("RSNA_DATA_ROOT") else None
OUT_DIR = Path(os.environ["RSNA_REPORT_LABEL_OUTPUT_DIR"]) if os.environ.get("RSNA_REPORT_LABEL_OUTPUT_DIR") else layout.output_dir

pre = preflight(layout=layout, data_root=DATA_ROOT, output_dir=OUT_DIR)
print(pre.message)
print("data_source=", pre.data_source)
print("scientific_evaluation=", pre.scientific_evaluation)
print("Competition top-level (not a DICOM walk):", pre.top_level_competition)

result = run_notebook_report_labeling(
    layout=layout,
    data_root=DATA_ROOT if DATA_ROOT is not None else pre.data_root,
    output_dir=OUT_DIR,
    data_source=None if DATA_ROOT is None else os.environ.get("RSNA_REPORT_LABEL_DATA_SOURCE"),
)

print("status=", result["status"])
print("DATA_SOURCE=", result["data_source"])
print("scientific_evaluation=", result["scientific_evaluation"])
print("claim_scientific_performance=", result.get("claim_scientific_performance"))
print("training_run=", result["training_run"], "folds_created=", result["folds_created"])
print("n_gold_studies=", result.get("n_gold_studies"), "n_report_only_studies=", result.get("n_report_only_studies"))
print("extractor_version=", result.get("extractor_version"))
if not result.get("scientific_evaluation"):
    print("FIXTURE OR NON-KAGGLE RUN: do not claim scientific gold-label performance.")


In [ ]:
# Display sanitized aggregates only. Never read or print private row-level files.
from pathlib import Path
import json

out = Path(result["output_dir"])
sanitized = out / "sanitized"
print("sanitized_dir=", sanitized)
print("DATA_SOURCE=", result["data_source"])
print("scientific_evaluation=", result["scientific_evaluation"])

summary_path = sanitized / "report_labeling_summary.md"
print(summary_path.read_text(encoding="utf-8")[:12000])

for name in (
    "gold_metrics.json",
    "language_audit.json",
    "weak_label_aggregates.json",
    "error_categories.json",
    "private_output_hashes.json",
    "report_label_manifest.json",
):
    path = sanitized / name
    print(f"\n=== {name} ===")
    text = path.read_text(encoding="utf-8")
    print(text[:4000])

assert "Report" not in (sanitized / "weak_label_aggregates.json").read_text()
print("private hashes recorded:", sorted((result.get("private_hashes") or {}).keys()))
